# Script per trasferire le annotazioni manuali tra file csv di rappresentazioni dei personaggi

## Per passare solo il tipo

In [ ]:
import csv

# 1. Carichiamo i dati originali in un dizionario per un accesso rapido
# Chiave: Titolo, Valore: Tipo
dati_orig = {}
with open("BoW_1000_personnages_mod.csv", mode="r", encoding="utf-8") as f_orig:
    reader = csv.reader(f_orig)
    for riga in reader:
        if riga: # Evita righe vuote
            dati_orig[riga[0]] = riga[3]

# 2. Leggiamo il file da aggiornare e scriviamo i risultati in un nuovo file
linee_aggiornate = []
with open("total_characters.csv", mode="r", encoding="utf-8") as f_dest:
    reader = csv.reader(f_dest)
    for riga in reader:
        titolo = riga[0]
        # Se il titolo è nel dizionario e la cella [3] è vuota o "0"
        if titolo in dati_orig and (riga[2] == "0" or riga[2] == ""):
            riga[2] = dati_orig[titolo]
        linee_aggiornate.append(riga)

# 3. Sovrascriviamo il file di destinazione con i dati modificati
with open("total_characters_annotato.csv", mode="w", encoding="utf-8", newline="") as f_output:
    writer = csv.writer(f_output)
    writer.writerows(linee_aggiornate)

print("Aggiornamento completato!")

Aggiornamento completato!


## Per passare anche il genere (autore e personaggio)

In [5]:
import csv

# 1. Carichiamo solo i dati necessari (colonne 1, 2, 3)
# Chiave: Titolo (col 0), Valore: Tupla (col 1, col 2, col 3)
dati_orig = {}
with open("Metadata/5_char_for_novel.csv", mode="r", encoding="utf-8") as f_orig:
    reader = csv.reader(f_orig)
    for riga in reader:
        if len(riga) > 3: # Verifica minima che esistano le colonne necessarie
            # Salviamo solo lo stretto indispensabile
            dati_orig[riga[0]] = (riga[1], riga[2], riga[3])

# 2. Leggiamo il file da aggiornare
linee_aggiornate = []
with open("tf-idf_personnages.csv", mode="r", encoding="utf-8") as f_dest:
    reader = csv.reader(f_dest)
    for riga in reader:
        titolo = riga[0]
        
        # Se il titolo esiste nel nostro archivio "leggero"
        if titolo in dati_orig:
            valori_nuovi = dati_orig[titolo] # Questa è la nostra tupla (col1, col2, col3)
            
            # Applichiamo i controlli colonna per colonna
            # Usiamo un ciclo per brevità se le colonne sono consecutive
            for i in range(1, 4): 
                # i va da 1 a 3. valori_nuovi[i-1] accede agli indici 0, 1, 2 della tupla
                if riga[i] == "":
                    riga[i] = valori_nuovi[i-1]
        
        linee_aggiornate.append(riga)

# 3. Scrittura finale
with open("tf-idf_personnages_ann.csv", mode="w", encoding="utf-8", newline="") as f_output:
    writer = csv.writer(f_output)
    writer.writerows(linee_aggiornate)

print("Aggiornamento selettivo completato!")

Aggiornamento selettivo completato!


## Per passare type

In [2]:
import pandas as pd

df_source = pd.read_csv("/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Memoire/Github_tesi/tf-idf_personnages_ann.csv")
df_dest = pd.read_csv("/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Memoire/Github_tesi/Notebooks/Textes/Supervised/BERT/total_characters.csv")

# Pulizia base (consigliata)
df_source["title"] = df_source["title"].str.strip()
df_dest["title"] = df_dest["title"].str.strip()

# Prendi solo le colonne utili e indicizza per title
df_source_subset = df_source[["title", "type","Gender"]].drop_duplicates("title")
df_source_subset = df_source_subset.set_index("title")

# Indice anche il dest
df_dest = df_dest.set_index("title")

# Aggiorna SOLO dove ci sono corrispondenze
df_dest.update(df_source_subset)

# Torna normale e salva
df_dest.reset_index().to_csv("total_characters_annotato.csv", index=False)